# QM9 fitted-featurizer cleanup

A compact companion to `qm9_e_homo_fix(2).ipynb`. Run these cells in order using your **buhito** Python kernel. No training, QM9 CSV, partition search, or full notebook execution is required.

This notebook clears only the molecule-specific `bi_transform_` locations and saves an **uncompressed** inference copy. It preserves the fitted graphlet vocabulary, feature order, configuration, and numerical precision. The original pickle is never overwritten. A cleaned file is published only after round-trip validation of the vocabulary, sparse features, and graphlet locations on three example molecules.

This is a targeted cleanup, not a general-purpose stripping of Buhito state. Validation covers the examples below; a large speedup is not guaranteed.


## 1. Imports and models directory

If you run this beside your original interpretability notebook, the default directory is `../models`. Edit `MODELS_DIR` if your saved featurizer is elsewhere. An existing `QM9_MODELS_DIR` environment variable takes precedence by default.


In [1]:
import gc
import os
import tempfile
from pathlib import Path
from time import perf_counter
from statistics import median

import joblib
from buhito.converters import smiles_to_nx

MODELS_DIR = Path(
    os.environ.get("QM9_MODELS_DIR", str(Path.cwd().parent / "models"))
).expanduser().resolve()
# Example override:
# MODELS_DIR = Path(r"C:\Users\statl\Documents\GitHub\buhito-interpretability\models")

ORIGINAL_PATH = MODELS_DIR / "buhito_featurizer.pkl"
CLEAN_PATH = MODELS_DIR / "buhito_featurizer_clean.pkl"
OVERWRITE_PREVIOUS_CLEAN = False  # Set True to replace an earlier cleaned copy.
TEST_SMILES = ["FC(F)(F)F", "O=C1NN=CN=C1", "COC12CC1OC2CO"]

if not ORIGINAL_PATH.is_file():
    raise FileNotFoundError(f"Edit MODELS_DIR: no featurizer found at {ORIGINAL_PATH}")
if ORIGINAL_PATH.resolve() == CLEAN_PATH.resolve():
    raise ValueError("The cleaned file must have a different path from the original.")
if CLEAN_PATH.exists() and not OVERWRITE_PREVIOUS_CLEAN:
    raise FileExistsError(
        f"{CLEAN_PATH} already exists. Set OVERWRITE_PREVIOUS_CLEAN=True or choose a new name."
    )
print("Original:", ORIGINAL_PATH)
print("Cleaned copy:", CLEAN_PATH)


Original: C:\Users\statl\Documents\GitHub\buhito-interpretability\models\buhito_featurizer.pkl
Cleaned copy: C:\Users\statl\Documents\GitHub\buhito-interpretability\models\buhito_featurizer_clean.pkl


## 2. Load and inspect the original

`bi_transform_` may contain graphlet-location dictionaries for molecules processed before the original object was saved. We record its entry count before running any new transforms. Other attributes are listed for inspection only; they are not deleted.


In [2]:
load_started = perf_counter()
featurizer = joblib.load(ORIGINAL_PATH)
original_load_seconds = perf_counter() - load_started
original_size = ORIGINAL_PATH.stat().st_size

print(f"Original file: {original_size / 1024**2:.2f} MiB")
print(f"Initial load: {original_load_seconds:.2f} seconds")
if not hasattr(featurizer, "bi_transform_"):
    raise AttributeError("No bi_transform_ attribute found. Inspect this Buhito version before cleaning.")
print("Saved location-cache type:", type(featurizer.bi_transform_).__name__)
try:
    print("Saved location-cache entries:", len(featurizer.bi_transform_))
except TypeError:
    print("Saved location cache has no length.")

for name, value in vars(featurizer).items():
    try:
        length = len(value)
    except TypeError:
        length = None
    print(f"{name}: type={type(value).__name__}, shape={getattr(value, 'shape', None)}, length={length}")

# Single-molecule inference settings used in the source notebook.
featurizer.n_jobs = 1
featurizer.verbose = False
vocabulary_hash = joblib.hash((featurizer.bit_ids_, featurizer.bit_sizes_))


Original file: 662.83 MiB
Initial load: 1672.01 seconds
Saved location-cache type: tuple
Saved location-cache entries: 26167
featurizer: type=BFSGraphletFeaturizer, shape=None, length=None
return_dense: type=bool, shape=None, length=None
return_float: type=bool, shape=None, length=None
n_jobs: type=int, shape=None, length=None
chunk_size: type=str, shape=None, length=4
verbose: type=int, shape=None, length=None
_cache: type=dict, shape=None, length=0
_in_fit_transform: type=bool, shape=None, length=None
n_bits_: type=int, shape=None, length=None
bit_ids_: type=list, shape=None, length=5370
bit_sizes_: type=ndarray, shape=(5370,), length=5370
bit_indices_: type=dict, shape=None, length=5370
n_unseen_: type=int, shape=None, length=None
X_unseen_: type=csr_matrix, shape=(26167, 195), length=None
bit_ids_unseen_: type=list, shape=None, length=195
bit_indices_unseen_: type=dict, shape=None, length=195
bit_sizes_unseen_: type=ndarray, shape=(195,), length=195
bi_fit_: type=tuple, shape=None,

## 3. Record reference outputs

Each SMILES is transformed separately, using explicit hydrogens as in model training. Graphlet occurrences are sorted for comparison while retaining duplicate instances. We compare all reported locations, not just nonzero feature counts.


In [3]:
def make_graph(smiles):
    converted = smiles_to_nx(smiles, add_hs=True, output_2d_pos=False)
    return converted[0] if isinstance(converted, (tuple, list)) else converted


def location_signature(bitinfo):
    return tuple(sorted(
        (
            repr(bit_id),
            tuple(sorted(
                tuple(sorted(int(atom) for atom in occurrence))
                for occurrence in occurrences
            )),
        )
        for bit_id, occurrences in bitinfo.items()
    ))


def snapshot(transformer, smiles):
    features = transformer.transform([make_graph(smiles)]).tocsr()
    if not hasattr(transformer, "bi_transform_") or len(transformer.bi_transform_) != 1:
        raise RuntimeError("Expected one graphlet-location dictionary after a one-molecule transform.")
    return features.copy(), location_signature(transformer.bi_transform_[0])


references = {smiles: snapshot(featurizer, smiles) for smiles in TEST_SMILES}
for smiles, (features, _) in references.items():
    print(f"Reference ready: {smiles}; shape={features.shape}; nonzero features={features.nnz}")


Reference ready: FC(F)(F)F; shape=(1, 5370); nonzero features=5
Reference ready: O=C1NN=CN=C1; shape=(1, 5370); nonzero features=68
Reference ready: COC12CC1OC2CO; shape=(1, 5370); nonzero features=83


## 4. Clean, save temporarily, reload, and validate

Only `bi_transform_` is cleared. A temporary file is round-trip tested before it is renamed to the cleaned output path. If a check fails, the temporary file is removed and no new cleaned output is published. Any earlier cleaned copy remains unchanged.

The original object is released before the cleaned object is loaded to avoid holding two large featurizers at once. The file on disk is saved **before** validation transforms, so it does not retain the test molecules' regenerated locations.


In [4]:
featurizer.bi_transform_ = []
handle, temporary_name = tempfile.mkstemp(
    prefix="featurizer_cleanup_", suffix=".pkl", dir=MODELS_DIR
)
os.close(handle)
temporary_path = Path(temporary_name)
clean = None

try:
    joblib.dump(featurizer, temporary_path, compress=0)
    del featurizer
    gc.collect()

    load_started = perf_counter()
    clean = joblib.load(temporary_path)
    clean_load_seconds = perf_counter() - load_started

    if len(clean.bi_transform_) != 0:
        raise AssertionError("The saved cleaned copy still contains graphlet-location entries.")
    if joblib.hash((clean.bit_ids_, clean.bit_sizes_)) != vocabulary_hash:
        raise AssertionError("The fitted vocabulary or feature ordering changed.")

    for smiles, (expected_features, expected_locations) in references.items():
        actual_features, actual_locations = snapshot(clean, smiles)
        if actual_features.shape != expected_features.shape:
            raise AssertionError(f"Feature shape changed: {smiles}")
        if (actual_features != expected_features).nnz != 0:
            raise AssertionError(f"Feature values changed: {smiles}")
        if actual_locations != expected_locations:
            raise AssertionError(f"Graphlet locations changed: {smiles}")
        print("Passed:", smiles)

    # Recheck the destination in case it was created during validation.
    if CLEAN_PATH.exists() and not OVERWRITE_PREVIOUS_CLEAN:
        raise FileExistsError(f"Refusing to replace existing cleaned copy: {CLEAN_PATH}")
    os.replace(temporary_path, CLEAN_PATH)
    clean_size = CLEAN_PATH.stat().st_size
    print("Validated cleaned copy saved:", CLEAN_PATH)
    print(f"Original: {original_size / 1024**2:.2f} MiB")
    print(f"Cleaned:  {clean_size / 1024**2:.2f} MiB")
    print(f"Size reduction: {100 * (1 - clean_size / original_size):.1f}%")
    print(f"Initial loads: original={original_load_seconds:.2f}s; cleaned={clean_load_seconds:.2f}s")
    print("These initial load times are not a controlled benchmark; OS disk caching can affect them.")
finally:
    temporary_path.unlink(missing_ok=True)
    # Validation transforms populate the in-memory cache again. Release that object.
    clean = None
    gc.collect()


Passed: FC(F)(F)F
Passed: O=C1NN=CN=C1
Passed: COC12CC1OC2CO
Validated cleaned copy saved: C:\Users\statl\Documents\GitHub\buhito-interpretability\models\buhito_featurizer_clean.pkl
Original: 662.83 MiB
Cleaned:  530.13 MiB
Size reduction: 20.0%
Initial loads: original=1672.01s; cleaned=777.41s
These initial load times are not a controlled benchmark; OS disk caching can affect them.


## 5. Optional repeated loading comparison

Set `RUN_BENCHMARK=True` to alternate loading the two files three times. Only one loaded object is retained at a time. This can take a while. Median times are reported, but the test primarily reflects warm filesystem-cache performance; do not interpret it as an unbiased cold-start benchmark.


In [5]:
RUN_BENCHMARK = False

if RUN_BENCHMARK:
    times = {"original": [], "cleaned": []}
    for repetition in range(3):
        order = [("original", ORIGINAL_PATH), ("cleaned", CLEAN_PATH)]
        if repetition % 2:
            order.reverse()
        for label, path in order:
            gc.collect()
            started = perf_counter()
            loaded_object = joblib.load(path)
            seconds = perf_counter() - started
            times[label].append(seconds)
            del loaded_object
            print(f"Run {repetition + 1}, {label}: {seconds:.2f}s")
    for label, values in times.items():
        print(f"Median {label} load: {median(values):.2f}s")
else:
    print("Benchmark skipped. Set RUN_BENCHMARK=True to run it.")


Benchmark skipped. Set RUN_BENCHMARK=True to run it.


## 6. Use the cleaned copy in the app

1. Stop the Shiny server.
2. Keep a backup of the original `buhito_featurizer.pkl`.
3. Copy `buhito_featurizer_clean.pkl` into the app's selected models folder under the expected name `buhito_featurizer.pkl`. If you use property-specific folders, use the correct matching featurizer for each model.
4. Restart the app to discard old in-memory model and molecule caches.
5. Compare a known molecule's prediction, contribution totals, and scores before and after. Disable Fast analysis and keep seed, initialization, k, beta, and iterations identical for a fixed-iteration comparison.

Exact feature vectors and unchanged feature ordering preserve the inputs to your unchanged trained models. The notebook checks features and locations; it does not load or test the HOMO/LUMO model files or run partition searches.

If size or loading time barely changes, `bi_transform_` was not the main cost. Do not remove other fitted attributes without checking how Buhito uses them. This notebook was checked for JSON structure and Python syntax. Its successful-cleanup and failure-cleanup paths passed end-to-end tests using synthetic featurizer fixtures. The actual Buhito transformer and saved artifact were unavailable during creation; execute the validation here in your own Buhito environment before adopting the output.
